**STRUCTURAL LAYERS**

In [1]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Two conv layers that keep the shape identical
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        # 1. Save the original input for the shortcut
        shortcut = x

        # 2. Pass through the processing layers
        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)

        # 3. THE STRUCTURAL ADDITION (The Residual Connection)
        out = out + shortcut

        out = self.relu(out)
        return out

dummy_x = torch.randn(1, 64, 32, 32)
res_block = ResidualBlock(channels=64)
print(f"Residual Output Shape: {res_block(dummy_x).shape}") # Stays (1, 64, 32, 32)

Residual Output Shape: torch.Size([1, 64, 32, 32])


In [2]:
class UNetShortcut(nn.Module):
    def __init__(self):
        super().__init__()
        # A processing layer that expects the concatenated size
        self.final_conv = nn.Conv2d(in_channels=128, out_channels=64, kernel_size=1)

    def forward(self, deep_features, shallow_features):
        # deep_features shape:   (Batch, 64, H, W)
        # shallow_features shape: (Batch, 64, H, W)

        # THE STRUCTURAL CONCATENATION (dim=1 is the channel dimension)
        # We are gluing them together along the channels
        glued_tensor = torch.cat((deep_features, shallow_features), dim=1)

        # glued_tensor shape is now: (Batch, 128, H, W)
        out = self.final_conv(glued_tensor)
        return out

dummy_deep = torch.randn(1, 64, 32, 32)
dummy_shallow = torch.randn(1, 64, 32, 32)
concat_block = UNetShortcut()
print(f"Concatenated Output Shape: {concat_block(dummy_deep, dummy_shallow).shape}")

Concatenated Output Shape: torch.Size([1, 64, 32, 32])


In [3]:
class GatedLinearUnit(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        # We split the input into two paths
        self.linear_data = nn.Linear(in_features, in_features)
        self.linear_gate = nn.Linear(in_features, in_features)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. The Data Path
        data = self.linear_data(x)

        # 2. The Gate Path (Values squeezed between 0 and 1)
        gate = self.sigmoid(self.linear_gate(x))

        # 3. THE STRUCTURAL MULTIPLICATION (Element-wise)
        # The gate dictates what percentage of the data is allowed through
        out = data * gate
        return out

dummy_signal = torch.randn(8, 128)
glu = GatedLinearUnit(128)
print(f"Gated Output Shape: {glu(dummy_signal).shape}")

Gated Output Shape: torch.Size([8, 128])
